In [60]:
from winogender_contextuality.utils import *
from winogender_contextuality.modeling.contextuality import *
from winogender_contextuality.config import * 
from winogender_contextuality.modeling.prompting import *
import ast
#from winogender_contextuality.modeling.meta_prompting import *
import matplotlib.pyplot as plt
import numpy as np

# Constants

In [22]:
questions = ['anaphora', 'pos', 'other_gender']

In [21]:
llama05 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_Llama-3.2-1B-Instruct_0.5_1350081025.ndjson")

In [101]:
gpt03 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_measurements_gpt-oss-20b_0.3.ndjson")
gpt05 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_measurements_gpt-oss-20b_0.5.ndjson")
gpt07 = load_ndjson(INTERIM_DATA_DIR / "metaprompting_measurements_gpt-oss-20b_0.7.ndjson")

# Functions

In [26]:
def get_meta_index(idx: int,
                   lst: list[dict]):

    """
    :param idx: pair index
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by index.
    """

    return [x for x in lst if x['index'] == idx]
    
def get_meta_question(q: str,
                      lst: list[dict]):

    """
    :param q: string key of the question
    :param lst: a list of MetaQA objects or equivalent dictionaries 

    Filters list of responses by question.
    """

    return [x for x in lst if x['question'] == q]

In [98]:
def get_meta_scores(lst: list[dict],
                   questions: list[str] = ['anaphora', 'pos', 'other_gender']):
    
    dic = {idx: {} for idx in range(60)}


    for idx in range(60):
        for q in questions:
            responses = get_meta_question(q, get_meta_index(idx, lst))
            n_correct = len([r for r in responses if r['response'] == r['answer'].replace("the ", "").replace("a ", "").strip().lower()])
            dic[idx][q] = n_correct/8 # as a fraction of best possible (including errors)

    return dic
        

In [93]:
def get_average_meta(dic: dict):

    response_arr = []
    for idx, dic in dic.items():
        response_arr.append(list(dic.values()))
    response_arr = np.array(response_arr)

    return np.mean(response_arr, axis=0)

In [78]:
def plot_response_dict(response_d):
    # Extract data
    indices = list(response_d.keys())
    categories = list(next(iter(response_d.values())).keys())
    colors = plt.cm.viridis(np.linspace(0, 1, len(categories)))  # color per category

    # Create plot
    plt.figure(figsize=(8, 5))

    for cat, color in zip(categories, colors):
        vals = [response_d[i][cat] for i in indices]
        plt.scatter(indices, vals, color=color, label=cat, s=50, alpha=0.7)

    # Labels and style
    plt.xlabel('Index')
    plt.ylabel('Score')
    plt.title('Response Dictionary Scatter Plot')
    plt.xticks(indices)
    plt.ylim(0, 1)
    plt.legend(title='Category')
    plt.grid(True, linestyle='--', alpha=0.6)

# Counting Errors

In [34]:
response_counts = {idx: {} for idx in range(60)}

for idx in range(60):
    for q in questions:
        responses = get_meta_question(q, get_meta_index(idx, llama05))
        n_not_null = len([r for r in responses if r['response'] != 'None'])
        response_counts[idx][q] = n_not_null

In [ ]:
# This is maybe not robust because there are instances where it says something like "BLANK" or something

# Counting Correct Responses

In [56]:
# Might have to adjust for spaces and lower case and all 
response_correctness = {idx: {} for idx in range(60)}

for idx in range(60):
    for q in questions:
        responses = get_meta_question(q, get_meta_index(idx, llama05))
        n_correct = len([r for r in responses if r['response'] == r['answer'].replace("the ", "").replace("a ", "").strip().lower()])
        response_correctness[idx][q] = n_correct/8 # as a fraction of best possible (including errors)
        

In [91]:
response_correctness_arr = []
for idx, dic in response_correctness.items():
    response_correctness_arr.append(list(dic.values()))
response_correctness_arr = np.array(response_correctness_arr)
print(f"Mean correctness Llama 3.2 1B: {np.mean(response_correctness_arr, axis=0)}")

Mean correctness Llama 3.2 1B: [0.53125 0.4     0.64375]


# GPT 

In [102]:
gpt03_scores = get_meta_scores(gpt03)
get_average_meta(gpt03_scores)

array([0.66041667, 1.        , 0.76666667])

In [100]:
gpt05_scores = get_meta_scores(gpt05)
get_average_meta(gpt05_scores)

array([0.65416667, 1.        , 0.76041667])

In [103]:
gpt07_scores = get_meta_scores(gpt07)
get_average_meta(gpt07_scores)

array([0.65625   , 1.        , 0.76041667])